# 05 — Stage 09: Resource-Aware Explainability

Orchestrates `src/xai/run_stage09.py` against the frozen, already-fitted `FIXED_ORIGIN`
model artifacts (`artifacts/models/BASE-FIXED-C1..C4-001.joblib`). No retraining, no split
change. All scientific logic lives in `src/xai/`; this notebook only orchestrates and
displays it — see `docs/experiments/STAGE09_RESOURCE_AWARE_XAI.md` for the full research note.

Headless equivalent: `python -m src.xai.run_stage09 --config configs/xai/stage09_resource_aware_xai_v1.yaml`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.xai.run_stage09 import run_stage09

## Run Stage 09

Re-running is safe: a new `EXP-XAI-000N` directory is created each time (see
`next_experiment_id` in `src/xai/run_stage09.py`), so nothing is silently overwritten. This cell
does not run automatically on notebook load in CI/tests — set `EXECUTE = True` to actually run it.

In [ ]:
EXECUTE = False  # flip to True to actually run Stage 09 from this notebook
config_path = REPO_ROOT / "configs/xai/stage09_resource_aware_xai_v1.yaml"

if EXECUTE:
    manifest = run_stage09(config_path)
else:
    print("EXECUTE=False; skipping run. See results/xai/ and artifacts/explanations/ for the", 
          "already-executed EXP-XAI-0001 run this notebook would reproduce.")

## Inspect the already-executed evidence

Loads the saved result tables directly — this cell is safe to run regardless of `EXECUTE` above.

In [ ]:
results_root = REPO_ROOT / "results/xai"
manifest_df = pd.read_csv(results_root / "stage09_manifest.csv")
manifest_df

In [ ]:
global_importance = pd.read_csv(results_root / "stage09_global_importance.csv")
(
    global_importance[
        (global_importance.method == "PERMUTATION_IMPORTANCE_MACRO_F1") & (global_importance.batch == "2")
    ]
    .sort_values(["model_id", "rank"])
    .groupby("model_id")
    .head(5)
)

In [ ]:
local_samples = pd.read_csv(results_root / "stage09_local_samples.csv")
local_samples["category"].value_counts()

## Not in scope for this notebook

Fidelity, stability, explanation latency, quantization, and hardware measurement are Stages
10-19 and are not computed here. `results/xai/stage09_fidelity_prep.csv` is prepared for Stage
10 to consume, but no fidelity score is calculated in this notebook.